# Grade 11 · Unit 1 — Bigger data and performance

> The table is generated here (**SYNTHETIC**, 500,000 rows) so the notebook is self-contained.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")
import time, sqlite3
rng = np.random.default_rng(1)
n = 500_000
big = pd.DataFrame({"station": rng.integers(0, 50, n), "hour": rng.integers(0, 24, n),
                    "pm25": rng.gamma(4, 15, n).round(1)})
print(big.shape, f"{big.memory_usage(deep=True).sum()/1e6:.1f} MB")

## Smaller types
**Predict:** by what factor can memory shrink?

In [ ]:
small = big.astype({"station": "int8", "hour": "int8", "pm25": "float32"})
before, after = big.memory_usage(deep=True).sum(), small.memory_usage(deep=True).sum()
assert after < before
print(f"{before/1e6:.1f} MB → {after/1e6:.1f} MB")

## Vectorised vs loop

In [ ]:
sample = big["pm25"].head(50_000)
t0 = time.perf_counter(); s1 = sum(x for x in sample); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); s2 = sample.sum(); t_vec = time.perf_counter() - t0
assert abs(s1 - s2) < 1e-3 * abs(s2)
print(f"loop {t_loop*1000:.1f} ms | vectorised {t_vec*1000:.2f} ms (timings vary by computer)")

## An index in SQLite

In [ ]:
con = sqlite3.connect(":memory:")
big.to_sql("readings", con, index=False)
def timed(sql):
    t0 = time.perf_counter(); out = con.execute(sql).fetchall(); return out, time.perf_counter() - t0
r1, t_no = timed("SELECT AVG(pm25) FROM readings WHERE station = 7")
con.execute("CREATE INDEX idx_station ON readings(station)")
r2, t_idx = timed("SELECT AVG(pm25) FROM readings WHERE station = 7")
assert abs(r1[0][0] - r2[0][0]) < 1e-9
print(f"without index {t_no*1000:.1f} ms | with index {t_idx*1000:.2f} ms")

**Make:** write two sentences: what you measured and why timings alone can mislead (repeat runs, other programs…).